<a href="https://colab.research.google.com/github/sinendoye/p1-sous-titreur-automatique/blob/transcription/notebooks/01_transcriptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1.** **Vérifier le GPU**

In [ ]:
!nvidia-smi

Mon Nov 23 12:13:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**2.** **Connecter ton Google Drive**

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**3. Indiquer le chemin de la vidéo et vérifier qu'elle existe**

In [ ]:
import os

VIDEO = "/content/drive/MyDrive/Atelier ia voix vidéo /WhatsApp Video 2026-11-23 at 12.11.32.mp4"   # ← à adapter

print("La vidéo existe :", os.path.exists(VIDEO))

La vidéo existe : True


**4. La fonction d'extraction du son**

In [ ]:
import subprocess

def extraire_audio(chemin_video, chemin_audio="/content/audio.wav"):
    """Extrait le son d'une vidéo en WAV mono 16 kHz, le format attendu par Whisper."""
    commande = [
        "ffmpeg",
        "-y",               # écrase le fichier audio s'il existe déjà
        "-i", chemin_video, # fichier d'entrée
        "-vn",              # "video no" : on ignore l'image
        "-ac", "1",         # 1 canal audio = mono
        "-ar", "16000",     # 16 000 échantillons par seconde
        chemin_audio,       # fichier de sortie
    ]
    subprocess.run(commande, check=True, capture_output=True)
    return chemin_audio

AUDIO = extraire_audio(VIDEO)
print("Audio créé :", AUDIO)

Audio créé : /content/audio.wav


**5. Vérification du résultat**

In [ ]:
from IPython.display import Audio

taille_ko = os.path.getsize(AUDIO) / 1024
print(f"Taille du fichier audio : {taille_ko:.0f} Ko")

Audio(AUDIO)

Taille du fichier audio : 1277 Ko


🪜 ÉTAPE 6 — Transcrire avec faster-whisper et la VAD
🎯 Objectif

Transformer l'audio en segments horodatés, au format convenu avec l'équipe :

audio.wav  →  faster-whisper + VAD  →  [{"start": 0.0, "end": 3.2, "text": "..."}, ...]


La VAD (Voice Activity Detection, détection d'activité vocale) repère les passages où quelqu'un parle et écarte les silences et la musique avant de les envoyer à Whisper. Sans elle, Whisper « invente » du texte sur les silences : c'est le piège signalé dans l'énoncé.

**6. Installation de faster-whisper**

In [ ]:
!pip install -q faster-whisper

**7. La fonction de transcription**

In [ ]:
from faster_whisper import WhisperModel
from scipy.io import wavfile          # 🆕 lit les fichiers WAV (déjà installé sur Colab)
import numpy as np                    # 🆕
import time

def charger_audio(chemin_audio):                              # 🆕
    """Lit un WAV 16 kHz et le convertit au format attendu par Whisper."""
    frequence, donnees = wavfile.read(chemin_audio)
    assert frequence == 16000, f"Fréquence inattendue : {frequence}"
    return donnees.astype(np.float32) / 32768.0

def transcrire(chemin_audio, taille_modele="small"):
    """Transcrit un audio en français. Renvoie (segments, durée de calcul en secondes)."""
    modele = WhisperModel(taille_modele, device="cuda", compute_type="float16")

    audio = charger_audio(chemin_audio)                       # 🆕

    debut = time.time()
    segments_bruts, info = modele.transcribe(
        audio,                                                # 🆕 les données, plus le chemin
        language="fr",
        vad_filter=True,
        beam_size=5,
    )
    segments = [
        {"start": round(s.start, 2), "end": round(s.end, 2), "text": s.text.strip()}
        for s in segments_bruts
    ]
    duree_calcul = round(time.time() - debut, 2)
    return segments, duree_calcul

**8. Lancer et afficher le résultat**

In [ ]:
segments, duree = transcrire(AUDIO, "small")

print(f"Temps de calcul : {duree} s — {len(segments)} segments\n")
for s in segments:
    print(f"[{s['start']:6.2f} → {s['end']:6.2f}]  {s['text']}")

Temps de calcul : 2.03 s — 9 segments

[  1.33 →   4.33]  Bonjour à tous, j'espère que vous allez bien, donc mon nom c'est Synda Esek,
[  4.33 →   9.33]  étudiant dans la référentielle intelligence artificielle ici à l'École du Côte Sonatel Académie.
[  9.33 →  15.33]  Donc je suis là pour vous parler un peu de l'intelligence artificielle parce qu'on sait tout ce que l'IA s'est imposé dans notre vie
[ 15.33 →  20.33]  et aujourd'hui au coeur de plus grande, de l'une des plus grandes évolutions technologiques de notre époque.
[ 20.33 →  25.33]  L'antenne qu'on se dirait comme une technologie réservée aux chercheurs ou bien aux plus grandes entreprises,
[ 25.33 →  28.33]  elle est désormais quelque chose qui est présente dans notre quotidien,
[ 28.33 →  33.33]  dans nos téléphones, dans nos moteurs de recherche, dans les établissements de santé partout
[ 33.33 →  35.33]  et même dans le secteur des transports.
[ 35.33 →  40.33]  C'est un secteur qui mérite de plus amples attention et qu'

🪜 ÉTAPE 7 — Comparer deux tailles de modèle
🎯 Objectif

L'énoncé demande de « comparer deux tailles de modèle Whisper en qualité et en temps de calcul ». Toi, tu mesures le temps. Adama mesurera la qualité (le WER) à partir des transcriptions que tu vas lui fournir.

On compare small (environ 240 millions de paramètres) et medium (environ 770 millions). Le modèle medium devrait mieux reconnaître les mots difficiles, mais il sera plus lent.

**9. Lancer les deux modèles**

In [ ]:
import pandas as pd

duree_audio = len(charger_audio(AUDIO)) / 16000   # durée en secondes

resultats = {}
for taille in ["small", "medium"]:
    print(f"Transcription avec {taille}...")
    segs, d = transcrire(AUDIO, taille)
    resultats[taille] = {"segments": segs, "duree": d}

tableau = pd.DataFrame([
    {
        "modèle": taille,
        "temps de calcul (s)": r["duree"],
        "facteur temps réel": round(r["duree"] / duree_audio, 3),
        "nb segments": len(r["segments"]),
    }
    for taille, r in resultats.items()
])
tableau

Transcription avec small...
Transcription avec medium...


,modèle,temps de calcul (s),facteur temps réel,nb segments
0,small,1.62,0.040,9
1,medium,3.32,0.081,5


**10. Comparer les textes**

In [ ]:
for taille, r in resultats.items():
    texte = " ".join(s["text"] for s in r["segments"])
    print(f"=== {taille} ===\n{texte}\n")

=== small ===
Bonjour à tous, j'espère que vous allez bien, donc mon nom c'est Synda Esek, étudiant dans la référentielle intelligence artificielle ici à l'École du Côte Sonatel Académie. Donc je suis là pour vous parler un peu de l'intelligence artificielle parce qu'on sait tout ce que l'IA s'est imposé dans notre vie et aujourd'hui au coeur de plus grande, de l'une des plus grandes évolutions technologiques de notre époque. L'antenne qu'on se dirait comme une technologie réservée aux chercheurs ou bien aux plus grandes entreprises, elle est désormais quelque chose qui est présente dans notre quotidien, dans nos téléphones, dans nos moteurs de recherche, dans les établissements de santé partout et même dans le secteur des transports. C'est un secteur qui mérite de plus amples attention et qu'on l'étudie encore plus.

=== medium ===
Bonjour à tous, j'espère que vous allez bien. Mon nom c'est Cyndole Sek, étudiant dans le référentiel intelligence artificielle ici à l'École du Côte de So

In [39]:
import json

DOSSIER_SORTIE = os.path.join(os.path.dirname(VIDEO), "outputs")
print("Les fichiers iront dans :", DOSSIER_SORTIE)
os.makedirs(DOSSIER_SORTIE, exist_ok=True)          # 🆕 crée le dossier s'il n'existe pas

for taille, r in resultats.items():
    chemin = f"{DOSSIER_SORTIE}/segments_{taille}.json"
    with open(chemin, "w", encoding="utf-8") as f:
        json.dump(r["segments"], f, ensure_ascii=False, indent=2)
    print("Enregistré :", chemin)

Les fichiers iront dans : /content/drive/MyDrive/Atelier ia voix vidéo /outputs
Enregistré : /content/drive/MyDrive/Atelier ia voix vidéo /outputs/segments_small.json
Enregistré : /content/drive/MyDrive/Atelier ia voix vidéo /outputs/segments_medium.json


**11. Sauvegarder pour l'équipe**

In [40]:
print("resultats existe :", "resultats" in globals())

resultats existe : True
